# Binomial distribution: exact formula and sampled evidence

Curriculum **4c**. Reference example, not a learner submission. For $n$ independent Bernoulli trials with the same success probability $p$, the number of successes $X$ has PMF

$$P(X=k)=\binom{n}{k}p^k(1-p)^{n-k},\quad k=0,\ldots,n.$$

This notebook uses $n=20$, $p=0.3$, validates the PMF against SciPy, then compares a seeded simulation with its theoretical mean. Trial independence and constant $p$ are model assumptions. Numerical checks at this parameter pair do not prove a theorem for all parameters.

Run from a fresh kernel after `uv sync --locked`. The original historical draft is preserved in `archive/BinomialDistribution.ipynb`.

In [ ]:
from math import comb

import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import binom

n = 20
p = 0.3
k = np.arange(n + 1)
probabilities = np.array([comb(n, int(j)) * p**j * (1 - p)**(n - j) for j in k])

# Absolute and relative tolerances for these moderate-size floating-point values.
assert np.all(probabilities >= 0)
np.testing.assert_allclose(probabilities.sum(), 1.0, rtol=0, atol=1e-12)
np.testing.assert_allclose(probabilities, binom.pmf(k, n, p), rtol=1e-12, atol=1e-15)
np.testing.assert_allclose(k @ probabilities, n * p, rtol=0, atol=1e-12)

In [ ]:
rng = np.random.default_rng(20261026)
sample_count = 100_000
samples = rng.binomial(n, p, size=sample_count)
empirical = np.bincount(samples, minlength=n + 1) / sample_count
expected_mean = n * p
standard_error = np.sqrt(n * p * (1 - p) / sample_count)
observed_mean = samples.mean()

# A generous stochastic diagnostic for this fixed seed, not a universal bound.
assert abs(observed_mean - expected_mean) < 6 * standard_error
print(f"Exact-model mean: {expected_mean:.3f}")
print(f"Sample mean: {observed_mean:.3f}; theoretical standard error: {standard_error:.5f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(k, probabilities, alpha=0.6, label="Binomial PMF")
ax.plot(k, empirical, "o", markersize=4, label="Seeded simulation")
ax.set(xlabel="Number of successes k", ylabel="Probability", title=f"n={n}, p={p}")
ax.legend()
fig.tight_layout()
plt.show()

## Checks and limits

The exact-model mean is $np$ and its variance is $np(1-p)$. The PMF calculation uses an exact integer coefficient followed by floating-point arithmetic. For large $n$ or extreme $p$, prefer stable library PMF/log-PMF routines and inspect underflow and conditioning.

The sampled mean fluctuates across seeds; six standard errors is a diagnostic tolerance, not a guarantee. Actual measurements could violate the assumed model. Try boundary cases $p=0$ and $p=1$ separately and explain why simulation still does not prove the general PMF formula.

Source pin: [MIT 18.05, Spring 2022](https://ocw.mit.edu/courses/18-05-introduction-to-probability-and-statistics-spring-2022/). The local code and checks are original. These outputs are preparation evidence only.